# Sprint 3 — Advanced challenger experiment (Colab launcher)

**Frozen representation:** `sentence-transformers/sentence-t5-base` — the embeddings already in
`sprint3_results/embeddings/` are reused; nothing is re-embedded and no API is called.

**Baseline:** Sentence-T5 + K-Means (configuration read from the Sprint 3 result files)

**Advanced challengers:**
- BERTopic (T5 embeddings → UMAP → HDBSCAN → c-TF-IDF topic representation)
- Leiden community detection (cosine kNN graph)
- Gaussian Mixture Model (diagonal covariance, Sprint 3 PCA space)

Principle: **complexity must earn its place** — a challenger replaces K-Means only with a meaningful gain and no
unacceptable cost. All logic lives in `sprint3_advanced_challengers.py`; this notebook only launches it and reads results.

Required in **My Drive/CAPESTONE PROJECT/**: this notebook, `sprint3_advanced_challengers.py`,
`sprint3_colab_experiment.py` (metric definitions are imported from it), `sprint3_task_input.csv`, `sprint3_results/`.
If the runtime disconnects, re-run cells 3–6: the script resumes from its checkpoints.

In [ ]:
# CELL 2 — GPU / environment (the experiment itself is CPU-bound; no model is loaded)
!nvidia-smi
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())

In [ ]:
# CELL 3 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 4 — paths + required existing files
import os

PROJECT_DIR = "/content/drive/MyDrive/CAPESTONE PROJECT"
SCRIPT = f"{PROJECT_DIR}/sprint3_advanced_challengers.py"
OUTPUT = f"{PROJECT_DIR}/sprint3_advanced_results"
ZIP = f"{PROJECT_DIR}/Sprint3_Advanced_Challenger_Results.zip"
for name, path in [("SCRIPT", SCRIPT), ("OUTPUT", OUTPUT), ("ZIP", ZIP)]:
    print(f"{name:7s} {path}")

required = [
    SCRIPT,
    f"{PROJECT_DIR}/sprint3_colab_experiment.py",
    f"{PROJECT_DIR}/sprint3_task_input.csv",
    f"{PROJECT_DIR}/sprint3_results/run_manifest.json",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_embeddings.npy",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_embedding_mapping.csv",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_meta.json",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_paired_text_embeddings.npy",
    f"{PROJECT_DIR}/sprint3_results/embeddings/sentence_t5_paired_text_mapping.csv",
    f"{PROJECT_DIR}/sprint3_results/evaluation/clustering_experiments.csv",
]
missing = [p for p in required if not os.path.isfile(p)]
for p in required:
    print(("OK      " if p not in missing else "MISSING ") + p)
if missing:
    raise FileNotFoundError("Required files are missing (see above). Embeddings are NOT regenerated.")

In [ ]:
# CELL 5 — install dependencies (BERTopic, UMAP, HDBSCAN, igraph + leidenalg, scientific stack).
# Packages Colab already provides are left as they are; no model is downloaded.
!pip install -q bertopic umap-learn hdbscan python-igraph leidenalg scikit-learn pandas numpy scipy matplotlib

In [ ]:
# CELL 6 — run the experiment
!python "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_challengers.py" \
  --project-dir "/content/drive/MyDrive/CAPESTONE PROJECT" \
  --output "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_advanced_results"

In [ ]:
# CELL 7 — verify results (read-only)
from pathlib import Path

if os.path.isfile(ZIP):
    print(f"ZIP: {ZIP}\nsize: {os.path.getsize(ZIP) / 1e6:.1f} MB")
else:
    print(f"ZIP NOT FOUND: {ZIP} - check the output of cell 6.")
out = Path(OUTPUT)
print("\nResult directories:")
for d in ["bertopic", "leiden", "gmm", "baseline", "evaluation", "figures"]:
    p = out / d
    n = len([f for f in p.rglob("*") if f.is_file()]) if p.exists() else 0
    print(f"  {d:11s} {'OK ' if p.exists() else 'MISSING'} {n} files")
for f in ["run_manifest.json", "advanced_challenger_report.md"]:
    print(f"  {f:30s} {'OK' if (out / f).exists() else 'MISSING'}")

In [ ]:
# CELL 8 — results preview (reads generated files; recalculates nothing)
import json
import pandas as pd
from IPython.display import Markdown, display

cmp_path = out / "evaluation" / "advanced_challenger_comparison.csv"
if cmp_path.exists():
    cmp = pd.read_csv(cmp_path)
    display(cmp)
    base = cmp[cmp["method"] == "kmeans"].iloc[0]
    print("CURRENT BASELINE:", base["pipeline"], "|", base["parameters"])
else:
    print("advanced_challenger_comparison.csv not found")

mf = out / "run_manifest.json"
if mf.exists():
    man = json.loads(mf.read_text())
    best = man.get("best_configurations", {})
    dec = man.get("replacement_decisions", {})
    qualified = [m for m, d in dec.items() if d.get("qualifies")]
    print("BEST CHALLENGER CONFIGS:", {m: c for m, c in best.items() if m != "kmeans"})
    print("CHALLENGERS THAT QUALIFIED TO REPLACE K-MEANS:", qualified or "none")
    print("FINAL RECOMMENDATION:", man.get("final_recommendation"))
    print("KEY REASON:", "; ".join(man.get("final_trace", [])))
    for m, d in dec.items():
        print(f"  {m}: gains={d.get('gains')} costs={d.get('costs')}")

rp = out / "advanced_challenger_report.md"
if rp.exists():
    display(Markdown(rp.read_text()))